# Uniform distribution: optimizer comparison

Distortion vs iteration for the optimizers you choose (**Lloyd**, **mean-field CLVQ**, **Newton–Raphson**, **Newton–Raphson (LM)**), starting from the **same** random centroids. Keys in code: `lloyd`, `mfclvq`, `nr`, `nrlm`.

Use **`n_iter >= 20`** when including **`nr`** or **`nrlm`**: both run 20 Lloyd warmup iterations, then `n_iter - 20` main steps, so the distortion series has length `n_iter`.

Run this notebook from the **repository root** (or set the kernel’s working directory there) so `univariate` imports resolve.

In [1]:
import sys
from pathlib import Path

# Ensure the repository root is on sys.path (works from repo root or from notebooks/)
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np

from bokeh.io import output_notebook, show
from bokeh.palettes import Category10
from bokeh.plotting import figure

from univariate.uniform_quantization import UniformVoronoiQuantization
from univariate.demos.optimizer_comparison import compare_methods, compare_nrlm_sweep

output_notebook()

np.random.seed(0)
N = 10
n_iter = 100

# Choose which methods to compare: any subset of ["lloyd", "mfclvq", "nr", "nrlm"]
methods = ["lloyd", "mfclvq", "nr", "nrlm"]

quantizer = UniformVoronoiQuantization()
initial = np.random.uniform(size=N)
curves = compare_methods(
    quantizer,
    initial,
    n_iter,
    methods=methods,
    nrlm_num_warmup_iterations=5,
)

p = figure(
    title="Uniform: distortion vs iteration (same initial centroids)",
    width=850,
    height=480,
    x_axis_label="Iteration",
    y_axis_label="Distortion",
    toolbar_location="above",
)

colors = Category10[10]
for i, (label, d) in enumerate(curves.items()):
    x = list(range(1, len(d) + 1))
    p.line(x, d, line_width=2, color=colors[i % len(colors)], legend_label=label)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.grid.grid_line_alpha = 0.25

show(p)

# Save an SVG snapshot for GitHub rendering.
# Bokeh SVG export requires a browser driver (chromedriver/geckodriver), so we fall back to Matplotlib.
from bokeh.io import export_svg

svg_dir = repo_root / "notebooks" / "_svg"
svg_dir.mkdir(parents=True, exist_ok=True)
svg_path_main = svg_dir / "uniform_comparison.svg"

try:
    p.output_backend = "svg"
    export_svg(p, filename=str(svg_path_main))
    print(f"Saved SVG (bokeh): {svg_path_main}")
except Exception as e:
    print(f"Bokeh SVG export failed ({type(e).__name__}): {e}")

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    for label, d in curves.items():
        ax.plot(range(1, len(d) + 1), d, label=label, linewidth=2)
    ax.set_title("Uniform: distortion vs iteration")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Distortion")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(svg_path_main, format="svg")
    plt.close(fig)
    print(f"Saved SVG (matplotlib): {svg_path_main}")

Loading BokehJS ...

/Users/montest/GitHub/deterministic-methods-optimal-quantization/univariate/voronoi_quantization.py:213: RuntimeWarning: invalid value encountered in multiply
  new_hessian = hessian + lambda_ * np.eye(len(centroids))


Bokeh SVG export failed (RuntimeError): Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.
Saved SVG (matplotlib): /Users/montest/GitHub/deterministic-methods-optimal-quantization/notebooks/_svg/uniform_comparison.svg


In [2]:
# Render the saved SVG (works on GitHub)
from IPython.display import SVG, display

if svg_path_main.exists():
    display(SVG(filename=str(svg_path_main)))
else:
    print(f"SVG not found: {svg_path_main}")

In [3]:
# NR+LM hyperparameter sweep
lambda_0_values = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]

sweep = compare_nrlm_sweep(
    quantizer,
    initial,
    n_iter=20,
    lambda_0_values=lambda_0_values,
    diagonal_term_types=("identity", "hessian"),
    nrlm_num_warmup_iterations=5,
)

p2 = figure(
    title="Uniform: NR+LM sweep over λ0 and diagonal term",
    width=850,
    height=480,
    x_axis_label="Iteration",
    y_axis_label="Distortion",
    toolbar_location="above",
)

colors = Category10[10]
dash_for_diag = {"identity": "solid", "hessian": "dashed"}

for lam_idx, lam0 in enumerate(lambda_0_values):
    color = colors[lam_idx % len(colors)]
    for diag in ("identity", "hessian"):
        d = sweep[(diag, float(lam0))]
        x = list(range(1, len(d) + 1))
        p2.line(
            x,
            d,
            line_width=2,
            color=color,
            line_dash=dash_for_diag[diag],
            legend_label=f"{diag}  λ0={lam0:g}",
        )

p2.legend.location = "top_right"
p2.legend.click_policy = "hide"
p2.grid.grid_line_alpha = 0.25

show(p2)

# Save an SVG snapshot for GitHub rendering.
# Bokeh SVG export requires a browser driver (chromedriver/geckodriver), so we fall back to Matplotlib.
from bokeh.io import export_svg

svg_path_sweep = svg_dir / "uniform_nrlm_sweep.svg"

try:
    p2.output_backend = "svg"
    export_svg(p2, filename=str(svg_path_sweep))
    print(f"Saved SVG (bokeh): {svg_path_sweep}")
except Exception as e:
    print(f"Bokeh SVG export failed ({type(e).__name__}): {e}")

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    for (diag, lam0), d in sweep.items():
        ax.plot(
            range(1, len(d) + 1),
            d,
            label=f"{diag}, lambda_0={lam0:g}",
            linewidth=2,
            linestyle=("-" if diag == "identity" else "--"),
        )
    ax.set_title("Uniform: NR+LM sweep")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Distortion")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(svg_path_sweep, format="svg")
    plt.close(fig)
    print(f"Saved SVG (matplotlib): {svg_path_sweep}")

Bokeh SVG export failed (RuntimeError): Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.
Saved SVG (matplotlib): /Users/montest/GitHub/deterministic-methods-optimal-quantization/notebooks/_svg/uniform_nrlm_sweep.svg


In [4]:
# Render the saved sweep SVG (works on GitHub)
from IPython.display import SVG, display

if svg_path_sweep.exists():
    display(SVG(filename=str(svg_path_sweep)))
else:
    print(f"SVG not found: {svg_path_sweep}")

In [5]:
for label, d in curves.items():
    print(f"{label:20s} final distortion = {d[-1]:.8f}")

Lloyd                final distortion = 0.00041751
Mean-field CLVQ      final distortion = 0.00134264
Newton–Raphson       final distortion = 0.00041667
Newton–Raphson (LM)  final distortion = 0.00041667
